In [ ]:
# ==============================
# LangChain + Gemini Agent Demo
# CAP6640 NLP Course Project
# ==============================

# gotta install these first so colab doesn't yell at us
!pip -q install -U langchain langchain-google-genai langchain-classic

# basic imports to make the magic happen
import os
import getpass
import time

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# secure api key input (don't dox the key on the recording lol)
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

# ==============================
# DEFINING THE TOOLS (The "Hands")
# ==============================

# TOOL 1: the RAG syllabus reader
@tool
def rag_syllabus_retriever(query: str) -> str:
    """
    Search the NLP class project requirements for relevant details.
    Use this when the user asks about due dates, team size, allowed topics, or submission format.
    """
    # visual cue so the prof actually sees the RAG happening on screen
    print("\n[System: Executing RAG Retrieval from Internal Document...]")

    project_doc = """
    CAP6640 Natural Language Processing Course Project
    Due: 4/5/2026
    Each project will be done by up to 3 students (individual work is also ok).
    You can either introduce an NLP relevant tool (software package) or survey on an NLP topic.
    Example tools: LLMs for NLP, AI agents for NLP, Mallet, Apache Lucene, GATE, NLTK, Carrot2, Deep learning tools
    Each team needs to prepare a 15-20 minute presentation.
    The recorded presentation can be uploaded to YouTube or other platforms and the video link submitted in Canvas.
    Each team only needs one submission listing all team members.
    """

    query_words = query.lower().split()
    matched_lines = []

    for line in project_doc.splitlines():
        clean_line = line.strip()
        if not clean_line:
            continue
        line_lower = clean_line.lower()
        if any(word in line_lower for word in query_words):
            matched_lines.append(clean_line)

    if matched_lines:
        return "\n".join(matched_lines[:10])

    return "Key facts: due date is 4/5/2026, teams can have up to 3 students, Canvas submission."

# TOOL 2: glossary lookup for the presentation
@tool
def nlp_term_lookup(term: str) -> str:
    """Look up a short definition for common NLP and LangChain terms."""
    print(f"\n[System: Accessing Glossary Database for '{term}'...]")
    glossary = {
        "nlp": "Natural Language Processing is the field of AI focused on understanding and generating human language.",
        "llm": "A large language model is trained on large amounts of text to understand and generate language.",
        "agent": "An AI agent uses a language model plus tools and can decide what actions to take to complete a task.",
        "langchain": "LangChain is an open-source framework for building LLM-powered applications and agents.",
        "rag": "Retrieval-Augmented Generation retrieves external information and uses it to generate more grounded answers."
    }
    return glossary.get(term.strip().lower(), f"No glossary entry found for '{term}'.")

# TOOL 3: calculator
@tool
def calculator(expression: str) -> str:
    """Safely evaluate basic arithmetic expressions."""
    print(f"\n[System: Executing Tool Use - Calculator for {expression}...]")
    allowed_chars = "0123456789+-*/(). "
    if any(ch not in allowed_chars for ch in expression):
        return "Error: only numbers and basic arithmetic symbols are allowed."
    try:
        return f"Result: {eval(expression, {'__builtins__': {}}, {})}"
    except Exception as e:
        return f"Error: {str(e)}"

# put all the tools in an array so the agent can see them
tools = [rag_syllabus_retriever, nlp_term_lookup, calculator]

# ==============================
# BUILDING THE AGENT (The "Brain")
# ==============================

# using 2.5 flash so we bypass any 404/deprecation errors
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# giving the agent its instructions
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an internal knowledge assistant for an NLP course project about LangChain. "
        "Use tools whenever they are helpful. "
        "If the user asks about class project rules, use the rag_syllabus_retriever tool. "
        "If the user asks for a definition, use the nlp_term_lookup tool. "
        "If math is needed, use the calculator tool."
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# initializing the actual agent executor
# (verbose=True is the secret sauce for the presentation visuals)
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# helper function to make the output look clean on the screen
def ask_agent(question: str):
    print("=" * 80)
    print("USER QUESTION:", question)
    print("=" * 80)
    response = agent_executor.invoke({"input": question})

    print("\nFINAL ANSWER:")
    output = response["output"]
    if isinstance(output, list):
        clean_text = "".join([item['text'] if isinstance(item, dict) and 'text' in item else str(item) for item in output])
        print(clean_text)
    else:
        print(output)
    print("\n")

# ==============================
# THE LIVE DEMO
# ==============================

# skipping the simple test questions for the live demo so we don't hit the free tier rate limit
ask_agent("What does RAG mean in NLP?")

print("\n[System: Pausing for 60 seconds to clear Free Tier API rate limits...]")
time.sleep(65) # Pauses the script for 65 seconds

# the grand finale demo question that triggers all three tools in one chain
ask_agent(
    "For our NLP class project, tell me the due date using the syllabus, explain what LangChain is using the glossary, "
    "and use the calculator to figure out how many minutes each person gets if a 15-minute presentation is split across 3 people."
)

USER QUESTION: What does RAG mean in NLP?


> Entering new AgentExecutor chain...

Invoking: `nlp_term_lookup` with `{'term': 'RAG'}`



[System: Accessing Glossary Database for 'RAG'...]
Retrieval-Augmented Generation retrieves external information and uses it to generate more grounded answers.RAG stands for Retrieval-Augmented Generation. It's a technique in NLP that retrieves external information and uses it to generate more grounded answers.

> Finished chain.

FINAL ANSWER:
RAG stands for Retrieval-Augmented Generation. It's a technique in NLP that retrieves external information and uses it to generate more grounded answers.



[System: Pausing for 60 seconds to clear Free Tier API rate limits...]
USER QUESTION: For our NLP class project, tell me the due date using the syllabus, explain what LangChain is using the glossary, and use the calculator to figure out how many minutes each person gets if a 15-minute presentation is split across 3 people.


> Entering new AgentExecutor cha